# SO3.2-02 — Inventário e validação da população total WorldPop

## Objetivo

Identificar, inventariar e avaliar as características técnicas da base WorldPop de população total que será utilizada no cálculo da exposição populacional à seca no SO3.2.

Esta etapa verifica:

- a coleção utilizada;
- a identificação dos dados correspondentes ao Brasil;
- o período temporal efetivamente disponível;
- a continuidade anual da série;
- a banda populacional;
- o sistema de referência espacial;
- a resolução nominal;
- a consistência da projeção e da grade entre os anos.

Esta etapa **não realiza o cruzamento com os produtos de seca do SO3.1**, não produz ainda o indicador de exposição e não define o tratamento dos anos posteriores ao último ano disponível na coleção.

## Fonte

**WorldPop Global Project Population Data**

Coleção no Google Earth Engine:

`WorldPop/GP/100m/pop`

A banda `population` representa a população estimada em cada célula da grade.

## 1. Configuração do ambiente

O Google Earth Engine é inicializado com o projeto utilizado no desenvolvimento do PRAIS. O Google Drive é montado apenas para registrar os produtos de QA/QC gerados neste notebook.

In [1]:
from google.colab import drive
from pathlib import Path

import ee
import pandas as pd

EE_PROJECT = "cursoqueimadas-503722"
WORLDPOP_TOTAL = "WorldPop/GP/100m/pop"

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/Cemaden")
PROJECT_ROOT = DRIVE_ROOT / "PRAIS4_SO3_BR"
SO32_ROOT = PROJECT_ROOT / "SO3.2"
SO32_LOGS = SO32_ROOT / "logs"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print(f"Google Earth Engine inicializado: {EE_PROJECT}")
print(f"Diretório de logs: {SO32_LOGS}")
print(f"Disponível: {SO32_LOGS.is_dir()}")

Mounted at /content/drive
Google Earth Engine inicializado: cursoqueimadas-503722
Diretório de logs: /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs
Disponível: True


## 2. Identificação da coleção e do Brasil

Primeiro é carregada a coleção completa. Em seguida, a propriedade `country` é consultada para identificar de forma explícita o código utilizado para o Brasil, evitando assumir previamente a codificação adotada pela coleção.

In [2]:
worldpop = ee.ImageCollection(WORLDPOP_TOTAL)

print("Coleção:", WORLDPOP_TOTAL)
print("Número total de imagens:", worldpop.size().getInfo())

countries = (
    worldpop
    .aggregate_array("country")
    .distinct()
    .sort()
    .getInfo()
)

brazil_candidates = [
    country
    for country in countries
    if (
        str(country).upper() == "BRA"
        or "BRAZIL" in str(country).upper()
        or "BRASIL" in str(country).upper()
    )
]

assert len(brazil_candidates) == 1, (
    "Não foi possível identificar de forma inequívoca "
    "o código do Brasil na coleção WorldPop."
)

BRAZIL_CODE = brazil_candidates[0]

print(f"Identificador utilizado para o Brasil: {BRAZIL_CODE}")

Coleção: WorldPop/GP/100m/pop
Número total de imagens: 5221
Identificador utilizado para o Brasil: BRA


## 3. Cobertura temporal da coleção brasileira

A coleção é filtrada pelo código do Brasil e ordenada pela propriedade `year`. São registrados o primeiro e o último ano disponíveis, o número de imagens e eventuais lacunas ou duplicações dentro da série observada.

In [3]:
worldpop_brazil = (
    worldpop
    .filter(ee.Filter.eq("country", BRAZIL_CODE))
    .sort("year")
)

years = [
    int(year)
    for year in worldpop_brazil
    .aggregate_array("year")
    .getInfo()
]

n_images = worldpop_brazil.size().getInfo()

expected_available_years = list(
    range(min(years), max(years) + 1)
)

missing_years = sorted(
    set(expected_available_years) - set(years)
)

duplicated_years = sorted({
    year
    for year in years
    if years.count(year) > 1
})

print(f"Imagens para o Brasil : {n_images}")
print(f"Primeiro ano          : {min(years)}")
print(f"Último ano            : {max(years)}")
print(f"Número de anos        : {len(years)}")
print(f"Anos ausentes         : {missing_years}")
print(f"Anos duplicados       : {duplicated_years}")

Imagens para o Brasil : 21
Primeiro ano          : 2000
Último ano            : 2020
Número de anos        : 21
Anos ausentes         : []
Anos duplicados       : []


**Resultado observado na validação:** a coleção brasileira contém 21 imagens anuais, cobrindo continuamente o período de 2000 a 2020. O tratamento necessário para atender ao período PRAIS posterior a 2020 será definido em etapa metodológica subsequente, sem alterar a caracterização da fonte realizada aqui.

## 4. Inventário das imagens anuais

Os identificadores `system:index` são registrados juntamente com o ano correspondente. Esse inventário fornece rastreabilidade sobre as imagens efetivamente selecionadas para o Brasil.

In [4]:
image_ids = (
    worldpop_brazil
    .aggregate_array("system:index")
    .getInfo()
)

worldpop_inventory = pd.DataFrame({
    "year": years,
    "system_index": image_ids,
})

worldpop_inventory

,year,system_index
0,2000,BRA_2000
1,2001,BRA_2001
2,2002,BRA_2002
3,2003,BRA_2003
4,2004,BRA_2004
5,2005,BRA_2005
6,2006,BRA_2006
7,2007,BRA_2007
8,2008,BRA_2008
9,2009,BRA_2009


## 5. Características da banda e da grade

A primeira imagem da série é utilizada como referência para identificar a banda disponível, o CRS, a resolução nominal e a transformação da projeção.

In [5]:
reference_image = ee.Image(worldpop_brazil.first())

bands = reference_image.bandNames().getInfo()

population_projection = (
    reference_image
    .select("population")
    .projection()
)

reference_crs = population_projection.crs().getInfo()
reference_scale = population_projection.nominalScale().getInfo()
reference_transform = population_projection.transform().getInfo()

print(f"Bandas              : {bands}")
print(f"CRS                 : {reference_crs}")
print(f"Resolução nominal   : {reference_scale:.3f} m")
print(f"Transformação       : {reference_transform}")

Bandas              : ['population']
CRS                 : EPSG:4326
Resolução nominal   : 92.766 m
Transformação       : PARAM_MT["Affine", 
  PARAMETER["num_row", 3], 
  PARAMETER["num_col", 3], 
  PARAMETER["elt_0_0", 0.0008333333300044304], 
  PARAMETER["elt_0_2", -73.989583022], 
  PARAMETER["elt_1_1", -0.0008333333300081173], 
  PARAMETER["elt_1_2", 5.264583514]]


## 6. Consistência da projeção entre os anos

Cada imagem anual é consultada separadamente. São registrados o país, o identificador da imagem, as bandas, o CRS, a resolução nominal e a transformação da projeção.

O objetivo é confirmar que a série anual brasileira utiliza uma estrutura espacial consistente antes das etapas de harmonização com o SPEI-12.

In [6]:
grid_records = []

for year in years:

    image = ee.Image(
        worldpop_brazil
        .filter(ee.Filter.eq("year", year))
        .first()
    )

    projection = (
        image
        .select("population")
        .projection()
    )

    info = ee.Dictionary({
        "year": year,
        "country": image.get("country"),
        "system_index": image.get("system:index"),
        "bands": image.bandNames(),
        "crs": projection.crs(),
        "nominal_scale_m": projection.nominalScale(),
        "transform": projection.transform(),
    }).getInfo()

    grid_records.append(info)

worldpop_grid = pd.DataFrame(grid_records)

### 6.1 Síntese estrutural

Como o objetivo é verificar consistência entre anos, o notebook apresenta uma síntese das propriedades distintas encontradas em vez de repetir toda a tabela técnica.

In [7]:
worldpop_summary = pd.Series({
    "Número de imagens":
        len(worldpop_grid),

    "Primeiro ano":
        min(years),

    "Último ano":
        max(years),

    "Anos ausentes":
        len(missing_years),

    "Anos duplicados":
        len(duplicated_years),

    "CRS distintos":
        worldpop_grid["crs"].nunique(),

    "Resoluções nominais distintas":
        worldpop_grid["nominal_scale_m"].nunique(),

    "Transformações distintas":
        worldpop_grid["transform"].astype(str).nunique(),
})

worldpop_summary

,0
Número de imagens,21
Primeiro ano,2000
Último ano,2020
Anos ausentes,0
Anos duplicados,0
CRS distintos,1
Resoluções nominais distintas,1
Transformações distintas,1


In [8]:
technical_profile = (
    worldpop_grid[
        [
            "country",
            "bands",
            "crs",
            "nominal_scale_m",
            "transform",
        ]
    ]
    .astype({"transform": str})
    .drop_duplicates()
    .reset_index(drop=True)
)

technical_profile

TypeError: unhashable type: 'list'

## 7. Testes automatizados de consistência

Os controles abaixo verificam os requisitos estruturais necessários para utilizar a série nas etapas subsequentes do SO3.2.

A série 2000–2020 é tratada aqui como a disponibilidade observada da fonte. A extensão da análise PRAIS até 2023 não é resolvida por imputação neste notebook.

In [9]:
assert SO32_LOGS.is_dir(), \
    "Diretório de logs do SO3.2 não encontrado."

assert n_images == len(years), \
    "O número de imagens não coincide com o número de anos inventariados."

assert len(missing_years) == 0, \
    "Há lacunas temporais dentro da série WorldPop disponível."

assert len(duplicated_years) == 0, \
    "Há mais de uma imagem para algum ano."

assert worldpop_grid["country"].nunique() == 1, \
    "Foi identificado mais de um código de país na série brasileira."

assert (worldpop_grid["country"] == BRAZIL_CODE).all(), \
    "Há imagem associada a código de país diferente do Brasil."

assert worldpop_grid["crs"].nunique() == 1, \
    "Foram identificados diferentes CRS na série."

assert worldpop_grid["nominal_scale_m"].nunique() == 1, \
    "Foram identificadas diferentes resoluções nominais."

assert worldpop_grid["transform"].astype(str).nunique() == 1, \
    "Foram identificadas diferentes transformações de grade."

assert (
    worldpop_grid["bands"]
    .apply(lambda x: x == ["population"])
    .all()
), "A estrutura de bandas difere do esperado."

print(
    "Controles estruturais da população total WorldPop atendidos."
)

Controles estruturais da população total WorldPop atendidos.


## 8. Registro dos produtos de QA/QC

São gravados no diretório de logs do SO3.2 dois produtos tabulares pequenos:

- `so32_02_worldpop_total_inventory.csv`: relação anual das imagens selecionadas;
- `so32_02_worldpop_total_grid.csv`: propriedades técnicas anuais da grade.

Esses arquivos documentam a coleção efetivamente observada no momento da execução do notebook.

In [10]:
inventory_output = (
    SO32_LOGS
    / "so32_02_worldpop_total_inventory.csv"
)

grid_output = (
    SO32_LOGS
    / "so32_02_worldpop_total_grid.csv"
)

worldpop_inventory.to_csv(
    inventory_output,
    index=False
)

worldpop_grid.to_csv(
    grid_output,
    index=False
)

print(f"Inventário anual : {inventory_output}")
print(f"Propriedades     : {grid_output}")

Inventário anual : /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs/so32_02_worldpop_total_inventory.csv
Propriedades     : /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs/so32_02_worldpop_total_grid.csv


## 9. Síntese da validação

A coleção `WorldPop/GP/100m/pop` foi identificada e filtrada para o Brasil por meio do código `BRA`.

Na execução utilizada para esta validação, a coleção brasileira apresenta **21 imagens anuais**, cobrindo continuamente o período de **2000 a 2020**, sem anos ausentes ou duplicados. Todas as imagens possuem exclusivamente a banda `population`.

A série apresenta estrutura espacial consistente entre os anos: um único sistema de referência (`EPSG:4326`), uma única resolução nominal (aproximadamente **92,77 m**) e uma única transformação de grade foram observados.

Não foram identificadas inconsistências estruturais que impeçam a utilização da população total WorldPop nas etapas subsequentes do SO3.2.

### Limite temporal da fonte

O período requerido pelo PRAIS alcança 2023, enquanto a série inventariada nesta coleção termina em 2020. **Nenhuma extrapolação, replicação ou imputação é realizada neste notebook.** O tratamento dos anos 2021–2023 será estabelecido explicitamente em etapa metodológica posterior.

### Situação da entrada

A população total WorldPop é considerada **tecnicamente adequada para o prosseguimento do SO3.2**, com o limite temporal da coleção claramente documentado.